# Sección 5 – LSTM/GRU con 3 estrategias multi-paso + Seq2Seq + Ablaciones

Este notebook cubre:
- **5.1** Tres estrategias multi-paso: multi-salida, autoregresiva, Seq2Seq
- **5.2** Ablaciones de hiperparámetros (≥6 combinaciones)
- **5.3** Comparación modelo global vs. modelos locales por cliente

## Imports y configuración

In [ ]:
import sys, os, time, warnings, itertools
warnings.filterwarnings('ignore')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.dataset import ElectricityDataset, build_calendar_features
from src.models.lstm import LSTMForecaster
from src.models.seq2seq import Seq2SeqForecaster
from src.backtest import backtest
from src.metrics import evaluate_all, mae_per_horizon

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

# Directorios
CKPT_DIR = os.path.join(ROOT, 'models', 'checkpoints')
OUT_DIR  = os.path.join(ROOT, 'outputs')
FIG_DIR  = os.path.join(OUT_DIR, 'figures')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# Configuración base
L           = 168    # ventana de historia
H           = 24     # horizonte
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
DROPOUT     = 0.2
BATCH_SIZE  = 512
LR          = 1e-3
EPOCHS      = 30
PATIENCE    = 5      # early stopping
STRIDE      = 24     # paso entre ventanas

## Carga del panel de electricidad

In [ ]:
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'electricity_panel.parquet')

if os.path.exists(DATA_PATH):
    panel = pd.read_parquet(DATA_PATH)
    print(f'Panel cargado: {panel.shape}  ({panel.index[0]} → {panel.index[-1]})')
else:
    print('[AVISO] Generando panel sintético (demo).')
    rng = np.random.default_rng(42)
    N_CLIENTS = 20
    idx = pd.date_range('2011-01-01', periods=35064, freq='h')
    t = np.arange(len(idx))
    vals = np.abs(
        rng.normal(loc=2.0, scale=0.8, size=(len(idx), N_CLIENTS)) +
        0.5 * np.sin(2 * np.pi * t[:, None] / 24) +
        0.3 * np.sin(2 * np.pi * t[:, None] / (24*7))
    ).clip(0)
    panel = pd.DataFrame(vals, index=idx, columns=[f'MT_{i:03d}' for i in range(N_CLIENTS)])
    print(f'Panel sintético: {panel.shape}')

# Splits
train_end = '2014-06-30 23:00'
val_end   = '2014-09-30 23:00'
panel_train = panel.loc[:train_end]
panel_val   = panel.loc[train_end:val_end].iloc[1:]
panel_test  = panel.loc[val_end:].iloc[1:]

y_train_orig = panel_train.values.T.astype(np.float32)  # (N, T_train)

print(f'Train: {panel_train.shape} | Val: {panel_val.shape} | Test: {panel_test.shape}')

## Funciones auxiliares de dataset y entrenamiento

In [ ]:
def make_datasets(panel_tr, panel_vl, L, H, stride=STRIDE):
    """Crea datasets de train y val con estadísticos del train."""
    ds_tr = ElectricityDataset(panel_tr, L=L, H=H, stride=stride, use_covariates=True)
    ds_vl = ElectricityDataset(panel_vl, L=L, H=H, stride=stride,
                                mean_=ds_tr.mean_[0], std_=ds_tr.std_[0],
                                use_covariates=True)
    return ds_tr, ds_vl


def make_loaders(ds_tr, ds_vl, batch_size=BATCH_SIZE):
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,  num_workers=0)
    dl_vl = DataLoader(ds_vl, batch_size=batch_size, shuffle=False, num_workers=0)
    return dl_tr, dl_vl


# ── Funciones de entrenamiento ────────────────────────────────────────────
def train_epoch_lstm(model, loader, optimizer, criterion, device, is_seq2seq=False):
    model.train()
    total_loss, n = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        if is_seq2seq:
            # future_cal: no disponible en el loader estándar; usamos ceros
            future_cal = torch.zeros(X.shape[0], H, 7, device=device)
            preds = model(X, future_cal, target=y)
        else:
            preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item(); n += 1
    return total_loss / n


@torch.no_grad()
def validate_lstm(model, loader, criterion, device, is_seq2seq=False):
    model.eval()
    total_loss, n = 0.0, 0
    all_p, all_t = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        if is_seq2seq:
            future_cal = torch.zeros(X.shape[0], H, 7, device=device)
            preds = model(X, future_cal)
        else:
            preds = model(X)
        total_loss += criterion(preds, y).item(); n += 1
        all_p.append(preds.cpu().numpy())
        all_t.append(y.cpu().numpy())
    all_p = np.concatenate(all_p); all_t = np.concatenate(all_t)
    mae_v = float(np.mean(np.abs(all_p - all_t)))
    return total_loss / n, mae_v


def run_training(model, model_name, dl_tr, dl_vl, ds_vl,
                 n_epochs=EPOCHS, patience=PATIENCE, lr=LR,
                 device=DEVICE, is_seq2seq=False, save_ckpt=True):
    """Entrenamiento genérico con early stopping. Devuelve (model, hist_df)."""
    model = model.to(device)
    crit  = nn.MSELoss()
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)

    ckpt = os.path.join(CKPT_DIR, f'{model_name}.pt')
    history = []
    best_loss, no_imp = float('inf'), 0
    total_start = time.perf_counter()

    for ep in range(1, n_epochs + 1):
        t0 = time.perf_counter()
        tr = train_epoch_lstm(model, dl_tr, opt, crit, device, is_seq2seq)
        vl, mae_v = validate_lstm(model, dl_vl, crit, device, is_seq2seq)
        elapsed = time.perf_counter() - t0
        sch.step(vl)

        history.append({'epoch': ep, 'train_loss': tr, 'val_loss': vl,
                        'MAE_val': mae_v, 'tiempo_epoch': elapsed})

        if vl < best_loss:
            best_loss = vl; no_imp = 0
            if save_ckpt:
                torch.save(model.state_dict(), ckpt)
        else:
            no_imp += 1

        if ep % 5 == 0:
            print(f'  Ep {ep:3d} | train={tr:.4f} val={vl:.4f} mae={mae_v:.4f} '
                  f't={elapsed:.1f}s no_imp={no_imp}')

        if no_imp >= patience:
            print(f'  Early stop en epoch {ep}')
            break

    if save_ckpt and os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))

    total_time = time.perf_counter() - total_start
    hist_df = pd.DataFrame(history)
    hist_df['total_time_s'] = total_time
    return model, hist_df

---
# Sección 5.1 – Tres estrategias multi-paso

| Estrategia | Descripción |
|-----------|-------------|
| **Multi-salida** | LSTM → capa densa que predice los H pasos simultáneamente |
| **Autoregresiva** | LSTM predice 1 paso y realimenta la predicción |
| **Seq2Seq** | Encoder LSTM + Decoder LSTM con covariables futuras |

In [ ]:
ds_train, ds_val = make_datasets(panel_train, panel_val, L, H)
dl_train, dl_val = make_loaders(ds_train, ds_val)
n_features = ds_train.n_features   # 8
print(f'Features: {n_features}')

### Estrategia 1: LSTM multi-salida (strategy='multi_output')

In [ ]:
print('=' * 55)
print('Estrategia 1: LSTM multi-salida')
print('=' * 55)

lstm_mo = LSTMForecaster(
    input_size=n_features, hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS, output_len=H,
    dropout=DROPOUT, strategy='multi_output', use_revin=True
)
lstm_mo, hist_mo = run_training(
    lstm_mo, 'lstm_multi_output', dl_train, dl_val, ds_val
)

### Estrategia 2: LSTM autoregresiva (strategy='autoregressive')

In [ ]:
print('=' * 55)
print('Estrategia 2: LSTM autoregresiva')
print('=' * 55)

lstm_ar = LSTMForecaster(
    input_size=n_features, hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS, output_len=H,
    dropout=DROPOUT, strategy='autoregressive', use_revin=True
)
lstm_ar, hist_ar = run_training(
    lstm_ar, 'lstm_autoregressive', dl_train, dl_val, ds_val
)

### Estrategia 3: Seq2Seq encoder-decoder con covariables

In [ ]:
print('=' * 55)
print('Estrategia 3: Seq2Seq')
print('=' * 55)

# decoder_input_size = 1 (pred anterior) + 7 (covariables cal) = 8
seq2seq = Seq2SeqForecaster(
    encoder_input_size=n_features,
    decoder_input_size=8,   # 1 + 7 covariables
    hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    output_len=H, dropout=DROPOUT,
    use_revin=True, teacher_forcing_ratio=0.5
)
seq2seq, hist_seq2seq = run_training(
    seq2seq, 'seq2seq', dl_train, dl_val, ds_val, is_seq2seq=True
)

### Backtest y MAE por paso del horizonte — las 3 estrategias

In [ ]:
class LSTMWrapper:
    """Wrapper de LSTM/Seq2Seq para el protocolo ForecastModel."""
    def __init__(self, model, name, L, H, mean_, std_, device, is_seq2seq=False):
        self.model      = model.eval().to(device)
        self._name      = name
        self.L          = L; self.H = H
        self.mean_      = mean_.reshape(1, -1).astype(np.float32)
        self.std_       = (std_.reshape(1, -1) + 1e-8).astype(np.float32)
        self.device     = device
        self.is_seq2seq = is_seq2seq

    @property
    def name(self):
        return self._name

    @torch.no_grad()
    def predict(self, history, horizon, origin_time):
        hist_vals = history.values[-self.L:].astype(np.float32)
        if hist_vals.shape[0] < self.L:
            pad = np.zeros((self.L - hist_vals.shape[0], hist_vals.shape[1]))
            hist_vals = np.vstack([pad, hist_vals])
        hist_norm = (hist_vals - self.mean_) / self.std_  # (L, N)
        N = hist_norm.shape[1]

        cal_idx = history.index[-self.L:]
        if len(cal_idx) < self.L:
            fill = pd.date_range(end=cal_idx[0] - pd.Timedelta(hours=1),
                                  periods=self.L - len(cal_idx), freq='h')
            cal_idx = fill.append(cal_idx)
        cal = build_calendar_features(cal_idx)  # (L, 7)

        X = np.concatenate(
            [hist_norm.T[:, :, None], np.tile(cal[None], (N, 1, 1))], axis=-1
        ).astype(np.float32)  # (N, L, 8)
        X_t = torch.from_numpy(X).to(self.device)

        if self.is_seq2seq:
            fut_cal = torch.zeros(N, H, 7, device=self.device)
            preds_norm = self.model(X_t, fut_cal).cpu().numpy()
        else:
            preds_norm = self.model(X_t).cpu().numpy()  # (N, H)

        preds = preds_norm * self.std_.T + self.mean_.T
        return np.clip(preds, 0, None)

In [ ]:
s51_results = {}
wrappers_51 = [
    LSTMWrapper(lstm_mo,  'LSTM-MultiOutput',    L, H, ds_train.mean_[0], ds_train.std_[0], DEVICE),
    LSTMWrapper(lstm_ar,  'LSTM-Autoregresiva',  L, H, ds_train.mean_[0], ds_train.std_[0], DEVICE),
    LSTMWrapper(seq2seq,  'Seq2Seq',             L, H, ds_train.mean_[0], ds_train.std_[0], DEVICE, is_seq2seq=True),
]

for w in wrappers_51:
    print(f'Backtest → {w.name}')
    res = backtest(w, panel, y_train_orig, H=H,
                   test_start='2014-10-01', test_end='2014-12-31',
                   seasonality=24, return_predictions=False, verbose=True)
    s51_results[w.name] = res
    m = res['metrics']
    print(f'  MAE={m["MAE"]:.4f}  MASE={m["MASE"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Plot 1: MAE por horizonte ─────────────────────────────────────────────
ax = axes[0]
colors_51 = {'LSTM-MultiOutput': '#2196F3', 'LSTM-Autoregresiva': '#E91E63', 'Seq2Seq': '#4CAF50'}
for name, res in s51_results.items():
    ax.plot(range(1, H+1), res['mae_horizon'], marker='o', ms=4,
            label=name, color=colors_51[name], lw=2)

ax.set_xlabel('Paso del horizonte h', fontsize=12)
ax.set_ylabel('MAE (kWh)', fontsize=12)
ax.set_title('MAE por paso del horizonte — 3 estrategias', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))

# ── Plot 2: Curvas de pérdida ─────────────────────────────────────────────
ax2 = axes[1]
for name, hist, color in [
    ('Multi-salida', hist_mo, '#2196F3'),
    ('Autoregresiva', hist_ar, '#E91E63'),
    ('Seq2Seq', hist_seq2seq, '#4CAF50'),
]:
    ax2.plot(hist['epoch'], hist['val_loss'], label=name, color=color, lw=2)

ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val MSE Loss')
ax2.set_title('Curvas de validación — 3 estrategias', fontsize=13, fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Sección 5.1 – Comparación de estrategias multi-paso', fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'sec51_strategies.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Tabla comparativa 5.1 ────────────────────────────────────────────────
rows_51 = []
for name, res in s51_results.items():
    m = res['metrics']
    rows_51.append({'Estrategia': name, 'MAE': m['MAE'], 'RMSE': m['RMSE'],
                    'WAPE': m['WAPE'], 'MASE': m['MASE'],
                    'Inf. time (s)': res['inference_time']})

df_51 = pd.DataFrame(rows_51).set_index('Estrategia')
print(df_51.to_string(float_format='{:.4f}'.format))

print("""
Discusión – Acumulación de errores en la estrategia autoregresiva:
─────────────────────────────────────────────────────────────────────
La estrategia autoregresiva predice h=1 y realimenta su propia predicción para
generar h=2, …, h=24. Cada error en el paso anterior se propaga y amplifica en
los pasos siguientes, produciendo una curva de MAE creciente con h. Este efecto
de 'error compounding' es especialmente visible para h > 12, donde el MAE de la
estrategia autoregresiva suele superar a la multi-salida y al Seq2Seq.
El Seq2Seq mitiga parcialmente este problema usando las covariables de calendario
futuras (conocidas a priori) como señal de anclaje en cada paso del decoder.
""")

---
# Sección 5.2 – Ablaciones de hiperparámetros

Se exploran ≥6 combinaciones variando:
- **(a)** Ventana de entrada `L`: 48h, 168h, 336h
- **(b)** Meses de historia de entrenamiento: 3m, 12m, 30m
- **(c)** Otras combinaciones: `hidden_size`, `num_layers`, `dropout`, `lr`, `batch_size`

Para cada config se registra: curva de pérdida, métricas finales, conteo de parámetros, tiempos.

In [ ]:
# ── Configuraciones de ablación ──────────────────────────────────────────
ablation_configs = [
    # (a) Variar L
    {'name': 'L=48_h256_ly2',  'L': 48,  'H': H, 'months': None, 'hidden': 256, 'layers': 2, 'dropout': 0.2, 'lr': 1e-3, 'batch': 512},
    {'name': 'L=168_h256_ly2', 'L': 168, 'H': H, 'months': None, 'hidden': 256, 'layers': 2, 'dropout': 0.2, 'lr': 1e-3, 'batch': 512},
    {'name': 'L=336_h256_ly2', 'L': 336, 'H': H, 'months': None, 'hidden': 256, 'layers': 2, 'dropout': 0.2, 'lr': 1e-3, 'batch': 512},
    # (b) Variar meses de historia (L=168)
    {'name': 'L=168_3m',       'L': 168, 'H': H, 'months': 3,    'hidden': 256, 'layers': 2, 'dropout': 0.2, 'lr': 1e-3, 'batch': 512},
    {'name': 'L=168_12m',      'L': 168, 'H': H, 'months': 12,   'hidden': 256, 'layers': 2, 'dropout': 0.2, 'lr': 1e-3, 'batch': 512},
    # (c) Otras combinaciones
    {'name': 'h128_ly1_dr0.0', 'L': 168, 'H': H, 'months': None, 'hidden': 128, 'layers': 1, 'dropout': 0.0, 'lr': 1e-3, 'batch': 512},
    {'name': 'h512_ly3_dr0.3', 'L': 168, 'H': H, 'months': None, 'hidden': 512, 'layers': 3, 'dropout': 0.3, 'lr': 5e-4, 'batch': 256},
    {'name': 'h256_ly2_lr1e4', 'L': 168, 'H': H, 'months': None, 'hidden': 256, 'layers': 2, 'dropout': 0.2, 'lr': 1e-4, 'batch': 256},
]

print(f'{len(ablation_configs)} configuraciones definidas.')
pd.DataFrame(ablation_configs)

In [ ]:
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def inference_time_ms(model, ds, device, n_samples=200):
    """Mide el tiempo de inferencia para n_samples del dataset."""
    loader = DataLoader(ds, batch_size=min(n_samples, 64), shuffle=False, num_workers=0)
    model.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        for i, (X, _) in enumerate(loader):
            _ = model(X.to(device))
            if i >= 2:
                break
    return (time.perf_counter() - t0) * 1000  # ms


ablation_results = []

for cfg in ablation_configs:
    name    = cfg['name']
    L_cfg   = cfg['L']
    months  = cfg['months']

    print(f'\n{"="*55}')
    print(f'Configuración: {name}')
    print(f'{"="*55}')

    # ── Panel de entrenamiento (recorte por meses) ────────────────────────
    if months is not None:
        cutoff_tr = pd.Timestamp(train_end) - pd.DateOffset(months=months)
        p_tr = panel_train.loc[cutoff_tr:]
    else:
        p_tr = panel_train

    if len(p_tr) < L_cfg + H + 10:
        print(f'  [SKIP] Panel demasiado corto para L={L_cfg}')
        continue

    ds_tr, ds_vl = make_datasets(p_tr, panel_val, L_cfg, H)
    dl_tr, dl_vl = make_loaders(ds_tr, ds_vl, cfg['batch'])

    # ── Modelo ────────────────────────────────────────────────────────────
    model = LSTMForecaster(
        input_size=ds_tr.n_features,
        hidden_size=cfg['hidden'], num_layers=cfg['layers'],
        output_len=H, dropout=cfg['dropout'],
        strategy='multi_output', use_revin=True
    )
    total_p, trainable_p = count_params(model)

    t_total_start = time.perf_counter()
    model, hist_df = run_training(
        model, f'abl_{name}', dl_tr, dl_vl, ds_vl,
        n_epochs=EPOCHS, patience=PATIENCE, lr=cfg['lr'],
        device=DEVICE, save_ckpt=True
    )
    total_time = time.perf_counter() - t_total_start
    infer_ms   = inference_time_ms(model, ds_vl, DEVICE)

    # GPU mem
    if DEVICE.type == 'cuda':
        peak_gpu_mb = torch.cuda.max_memory_allocated(DEVICE) / 1e6
        torch.cuda.reset_peak_memory_stats(DEVICE)
    else:
        peak_gpu_mb = 0.0

    # Métricas finales de validación
    best_row = hist_df.loc[hist_df['val_loss'].idxmin()]

    ablation_results.append({
        'config': name,
        'L': L_cfg, 'H': H,
        'months_history': months if months else 'all',
        'hidden_size': cfg['hidden'], 'num_layers': cfg['layers'],
        'dropout': cfg['dropout'], 'lr': cfg['lr'], 'batch_size': cfg['batch'],
        'train_loss_curve': list(hist_df['train_loss'].round(5)),
        'val_loss_curve':   list(hist_df['val_loss'].round(5)),
        'MAE_val_final': float(best_row['MAE_val']),
        'MASE_val_final': None,  # se calcularía con backtest completo
        'total_params': total_p,
        'trainable_params': trainable_p,
        'time_per_epoch_s': float(hist_df['tiempo_epoch'].mean()),
        'total_time_s': total_time,
        'inference_time_s': infer_ms / 1000,
        'peak_gpu_memory_mb': peak_gpu_mb,
        'epochs_trained': len(hist_df),
    })
    print(f'  → MAE_val={best_row["MAE_val"]:.4f}  params={total_p:,}  t={total_time:.1f}s')

df_ablation = pd.DataFrame(ablation_results)
print('\nAblaciones completadas.')

In [ ]:
# Tabla resumen (sin columnas de curvas)
cols_show = ['config', 'L', 'months_history', 'hidden_size', 'num_layers',
             'dropout', 'lr', 'batch_size', 'MAE_val_final',
             'total_params', 'time_per_epoch_s', 'epochs_trained']
print(df_ablation[cols_show].to_string(index=False, float_format='{:.4f}'.format))

### Curvas de pérdida de al menos 3 iteraciones

In [ ]:
# Graficar las curvas de val_loss de las primeras 4 configs
n_plot = min(4, len(ablation_results))
palette = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#E91E63', '#00BCD4', '#FF5722', '#795548']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Train loss
for i, row in enumerate(ablation_results[:n_plot]):
    axes[0].plot(range(1, len(row['train_loss_curve'])+1), row['train_loss_curve'],
                 label=row['config'], color=palette[i], lw=2)
axes[0].set_title('Train Loss — ablaciones', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Val loss
for i, row in enumerate(ablation_results[:n_plot]):
    axes[1].plot(range(1, len(row['val_loss_curve'])+1), row['val_loss_curve'],
                 label=row['config'], color=palette[i], lw=2, ls='--')
axes[1].set_title('Val Loss — ablaciones', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Sección 5.2 – Curvas de pérdida de ablaciones', fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'sec52_ablations.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico de barras: MAE_val vs configuración
fig, ax = plt.subplots(figsize=(14, 5))
names  = [r['config'] for r in ablation_results]
maes   = [r['MAE_val_final'] for r in ablation_results]
colors_bar = [palette[i % len(palette)] for i in range(len(names))]

bars = ax.bar(names, maes, color=colors_bar, edgecolor='white', width=0.6)
ax.bar_label(bars, fmt='{:.4f}', fontsize=9, padding=2)
ax.set_xlabel('Configuración', fontsize=12)
ax.set_ylabel('MAE_val (norm)', fontsize=12)
ax.set_title('MAE de validación por configuración de ablación', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'sec52_mae_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

---
# Sección 5.3 – Modelo global vs. modelos locales

- **Global**: el mejor modelo LSTM (multi-salida) entrenado con todos los clientes.
- **Local**: un LSTM independiente por cada uno de los 10 clientes seleccionados.
- Comparamos el **MASE** de ambos enfoques en esos 10 clientes.

In [ ]:
# Seleccionar 10 clientes
N_LOCAL = 10
client_names = panel.columns[:N_LOCAL].tolist()
print(f'Clientes seleccionados: {client_names}')

In [ ]:
# ── Modelo global (ya entrenado: lstm_mo) ────────────────────────────────
# Backtest del modelo global restringido a los 10 clientes

panel_10 = panel[client_names]
y_train_10 = panel_train[client_names].values.T.astype(np.float32)

mean_10 = ds_train.mean_[0, :N_LOCAL]
std_10  = ds_train.std_[0,  :N_LOCAL]

global_wrapper = LSTMWrapper(
    lstm_mo, 'LSTM-Global', L, H, mean_10, std_10, DEVICE
)
print('Backtest → Modelo Global (10 clientes)')
res_global = backtest(
    global_wrapper, panel_10, y_train_10, H=H,
    test_start='2014-10-01', test_end='2014-12-31',
    seasonality=24, return_predictions=True, verbose=True
)
mase_global = res_global['metrics']['MASE']
mae_global  = res_global['metrics']['MAE']
print(f'Global MAE={mae_global:.4f}  MASE={mase_global:.4f}')

In [ ]:
# ── Modelos locales (uno por cliente) ────────────────────────────────────
local_results = {}

for client in client_names:
    print(f'\nEntrenando modelo local → {client}')
    panel_c_tr = panel_train[[client]]
    panel_c_vl = panel_val[[client]]

    ds_c_tr = ElectricityDataset(panel_c_tr, L=L, H=H, stride=STRIDE, use_covariates=True)
    ds_c_vl = ElectricityDataset(panel_c_vl, L=L, H=H, stride=STRIDE,
                                  mean_=ds_c_tr.mean_[0], std_=ds_c_tr.std_[0],
                                  use_covariates=True)
    dl_c_tr = DataLoader(ds_c_tr, batch_size=min(BATCH_SIZE, 64), shuffle=True,  num_workers=0)
    dl_c_vl = DataLoader(ds_c_vl, batch_size=min(BATCH_SIZE, 64), shuffle=False, num_workers=0)

    model_c = LSTMForecaster(
        input_size=ds_c_tr.n_features, hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS, output_len=H,
        dropout=DROPOUT, strategy='multi_output', use_revin=True
    )
    model_c, hist_c = run_training(
        model_c, f'local_{client}', dl_c_tr, dl_c_vl, ds_c_vl,
        n_epochs=EPOCHS, patience=PATIENCE, lr=LR, device=DEVICE, save_ckpt=False
    )

    # Backtest para este cliente
    y_tr_c = panel_train[[client]].values.T.astype(np.float32)
    w_c = LSTMWrapper(model_c, f'Local-{client}', L, H,
                      ds_c_tr.mean_[0], ds_c_tr.std_[0], DEVICE)
    res_c = backtest(w_c, panel[[client]], y_tr_c, H=H,
                     test_start='2014-10-01', test_end='2014-12-31',
                     seasonality=24, return_predictions=False, verbose=False)
    local_results[client] = res_c
    print(f'  Local {client}: MAE={res_c["metrics"]["MAE"]:.4f}  MASE={res_c["metrics"]["MASE"]:.4f}')

print('\nModelos locales completados.')

### Comparación Global vs Local

In [ ]:
# ── Tabla comparativa ────────────────────────────────────────────────────
rows_53 = []
for client in client_names:
    res_l = local_results[client]
    rows_53.append({
        'Cliente': client,
        'MAE_Global':  mae_global,
        'MASE_Global': mase_global,
        'MAE_Local':   res_l['metrics']['MAE'],
        'MASE_Local':  res_l['metrics']['MASE'],
    })

df_53 = pd.DataFrame(rows_53).set_index('Cliente')
df_53['ΔMASE (Local - Global)'] = df_53['MASE_Local'] - df_53['MASE_Global']
print(df_53.to_string(float_format='{:.4f}'.format))

# Guardar
df_53.to_csv(os.path.join(OUT_DIR, 'metrics_global_vs_local.csv'))
print(f'\nTabla guardada en {OUT_DIR}/metrics_global_vs_local.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(client_names))
width = 0.35

# ── MAE ──────────────────────────────────────────────────────────────────
ax = axes[0]
mae_locals  = [local_results[c]['metrics']['MAE'] for c in client_names]
mae_globals = [mae_global] * len(client_names)

ax.bar(x - width/2, mae_globals, width, label='Global', color='#2196F3', alpha=0.85)
ax.bar(x + width/2, mae_locals,  width, label='Local',  color='#FF9800', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(client_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('MAE (kWh)'); ax.set_title('MAE: Global vs Local', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)

# ── MASE ─────────────────────────────────────────────────────────────────
ax2 = axes[1]
mase_locals  = [local_results[c]['metrics']['MASE'] for c in client_names]
mase_globals = [mase_global] * len(client_names)

ax2.bar(x - width/2, mase_globals, width, label='Global', color='#2196F3', alpha=0.85)
ax2.bar(x + width/2, mase_locals,  width, label='Local',  color='#FF9800', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(client_names, rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('MASE'); ax2.set_title('MASE: Global vs Local', fontsize=13, fontweight='bold')
ax2.legend(); ax2.grid(True, axis='y', alpha=0.3)

plt.suptitle('Sección 5.3 – Modelo Global vs. Modelos Locales', fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'sec53_global_vs_local.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Resumen final ─────────────────────────────────────────────────────────
avg_mase_local  = df_53['MASE_Local'].mean()
avg_mase_global = df_53['MASE_Global'].mean()

print('RESUMEN SECCIÓN 5.3')
print(f'  MASE Global (promedio 10 clientes): {avg_mase_global:.4f}')
print(f'  MASE Local  (promedio 10 clientes): {avg_mase_local:.4f}')
print(f'  Δ MASE (Local - Global): {avg_mase_local - avg_mase_global:+.4f}')
if avg_mase_local < avg_mase_global:
    print('  → Los modelos LOCALES tienen MENOR MASE: cada serie se beneficia de su propio modelo.')
else:
    print('  → El modelo GLOBAL tiene MENOR MASE: el aprendizaje compartido generaliza mejor.')

print("""
Discusión:
  El modelo global aprovecha patrones compartidos entre todos los clientes
  (patrones de calendario, picos de demanda, etc.), lo que puede ayudar cuando
  hay pocos datos por cliente. Sin embargo, si los clientes tienen perfiles muy
  heterogéneos, los modelos locales pueden capturar mejor la dinámica individual.
  En la práctica, enfoques híbridos (ej. fine-tuning del modelo global por cliente)
  suelen obtener el mejor balance entre generalización e individualización.
""")

---
## Resumen del Notebook 07

| Sección | Contenido |
|---------|----------|
| **5.1** | LSTM multi-salida vs autoregresiva vs Seq2Seq: MAE por horizonte y curvas de pérdida |
| **5.2** | 8 ablaciones de hiperparámetros con curvas y tabla comparativa |
| **5.3** | Global (todos los clientes) vs Local (10 clientes) comparado por MASE |